<a href="https://colab.research.google.com/github/matinbyrml/Neural-Networks/blob/main/TrainingCustomModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# if available use the gpu
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device} device')



Using cuda device


In [2]:
# custom network class, we inherit all critical functionalities from pytorch
class NeuralNetwork(nn.Module):
  # we define layers and sub modules we will use for the network
  def __init__(self):
    super(NeuralNetwork, self).__init__()
    # convert multi d inpit to 1 dimensional vetor of features, 28x28 image to 784 features for ex
    self.flatten = nn.Flatten()
    # sequential stack of the operations
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(3*64*64, 512), # 1st FN, with output of 512, input 784, we learn weights and biases here
        nn.ReLU(), # activation function
        nn.Linear(512, 512), # hidden layer
        nn.ReLU(),
        nn.Linear(512,200), # output linear layer, with 10 output features (for ex, for 10 classes like in FashionMNIST)
        # nn.ReLU()
    )
  # we define how our input data flows thru layers.
  # this method is not called directly
  def forward(self, x):
    # reshape the tensor
    x = self.flatten(x)
    # flattened tensor is passed thru layers
    logits = self.linear_relu_stack(x)
    # produce logits ( raw scores that we will feed our loss function with )
    return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=12288, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=200, bias=True)
    (5): ReLU()
  )
)


In [11]:
# loss function
# we are using it for a classification problem,
# cross entropy is a common choice
# combination of LogSoftMax and NLL
loss_fn = nn.CrossEntropyLoss()
# optimizer
# we are using it to adjust model's weights and biases
# model params -> all learnable params, weights and biases
# optimizer updates these two
# learning rate lr, hyperparameter
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [3]:
def train_loop(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  # helps to prevent overfitting and uses batch normlz
  model.train()
  for batch, (X, y) in enumerate(dataloader):
    #computer the prediction and the loss
    X = X.to(device)
    y = y.to(device)
    pred = model(X)
    loss = loss_fn(pred, y)

    # backpropogation, zero out the accumulate gradients
    optimizer.zero_grad()
    # calculate the gradients of the loss wrt params
    loss.backward()
    # update the params based on the calculated gradients
    optimizer.step()

    if batch % 100 == 0:
      loss, current = loss.item(), batch * len(X)
      print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [4]:
def test_loop(dataloader, model, loss_fn):
  # total num of samples in the dataset
  size = len(dataloader.dataset)
  test_loss, correct = 0, 0
  # disable the gradient tracking
  # no learning (not updating the weights), just evaluationg
  with torch.no_grad():
    for X, y in dataloader:
      X = X.to(device)
      y = y.to(device)
      # move the input data thru model and get raw scores (logits)
      pred = model(X)
      # accumulate th eloss for the current batch
      test_loss += loss_fn(pred, y).item()
      # calculate the total correct predictions in the batch
      # pred.argmax(1) is the highest logit for each sample
      # y is the true label
      # and convert boolean to a number
      # sum all true labels (all 1s)
      # extract the number
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()
  #normalize
  test_loss /= size
  correct /= size
  # see what is going on
  print(f"Test Error: \n Accuracy {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [13]:
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip -q tiny-imagenet-200.zip

URL transformed to HTTPS due to an HSTS policy
--2025-11-26 20:01:39--  https://cs231n.stanford.edu/tiny-imagenet-200.zip
Resolving cs231n.stanford.edu (cs231n.stanford.edu)... 171.64.64.64
Connecting to cs231n.stanford.edu (cs231n.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 248100043 (237M) [application/zip]
Saving to: ‘tiny-imagenet-200.zip.2’

tiny-imagenet-200.z 100%[===================>] 236.61M  11.2MB/s    in 23s     

2025-11-26 20:02:02 (10.4 MB/s) - ‘tiny-imagenet-200.zip.2’ saved [248100043/248100043]



In [14]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor

dataset = ImageFolder(root="tiny-imagenet-200/train", transform=ToTensor())
#see total number of samples
#even dataset contains the train folder, ImageFolder
#implements the __len__ method such that, it looks at also in
#subfolders
print(f"Total number of samples: {len(dataset)}")



Total number of samples: 100000


In [7]:
# see all the available classes, ImageFolder finds
# same info exists also in wnids.txt file
print(dataset.classes)
# we can see how many classes it contains
# alternatively we can count the number of rows in wnids,txt
print(len(dataset.classes))



['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750', 'n01784675', 'n01855672', 'n01882714', 'n01910747', 'n01917289', 'n01944390', 'n01945685', 'n01950731', 'n01983481', 'n01984695', 'n02002724', 'n02056570', 'n02058221', 'n02074367', 'n02085620', 'n02094433', 'n02099601', 'n02099712', 'n02106662', 'n02113799', 'n02123045', 'n02123394', 'n02124075', 'n02125311', 'n02129165', 'n02132136', 'n02165456', 'n02190166', 'n02206856', 'n02226429', 'n02231487', 'n02233338', 'n02236044', 'n02268443', 'n02279972', 'n02281406', 'n02321529', 'n02364673', 'n02395406', 'n02403003', 'n02410509', 'n02415577', 'n02423022', 'n02437312', 'n02480495', 'n02481823', 'n02486410', 'n02504458', 'n02509815', 'n02666196', 'n02669723', 'n02699494', 'n02730930', 'n02769748', 'n02788148', 'n02791270', 'n02793495', 'n02795169', 'n02802426', 'n02808440', 'n02814533', 'n02814860', 'n02815834', 'n02823428', 'n02837789', 'n02841315', 'n02843684'

In [ ]:
# tiny image dataset is organized so we can use ImageFolder

In [8]:
from torch.utils.data import DataLoader
batch_size = 64
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [15]:
# time to load validation data set
# we need to reead the val_annotation.txt
# Column 1: Image filename (e.g., val_0.JPEG)
# Column 2: Class ID (e.g., n03444034)
# Columns 3-6: Bounding box coordinates (which we can ignore for simple classification).
import os
import shutil

# 1. Define paths
val_dir = 'tiny-imagenet-200/val'
img_dir = os.path.join(val_dir, 'images')
annot_file = os.path.join(val_dir, 'val_annotations.txt')

# 2. Read annotations and create a mapping
with open(annot_file, 'r') as f:
    annotations = f.readlines()

# 3. Process each image/annotation pair
for line in annotations:
    parts = line.split('\t')
    img_name = parts[0]
    class_id = parts[1] # e.g., 'n03444034'

    # A. Create the destination class folder if it doesn't exist
    class_folder = os.path.join(val_dir, class_id)
    os.makedirs(class_folder, exist_ok=True)

    # B. Move the image file to the new class folder
    source_path = os.path.join(img_dir, img_name)
    destination_path = os.path.join(class_folder, img_name)

    # TinyImageNet validation structure has another subfolder called 'images'
    # inside each class folder. We need to create that too.
    os.makedirs(os.path.join(class_folder, 'images'), exist_ok=True)
    destination_path = os.path.join(class_folder, 'images', img_name)


    # Move the image
    if os.path.exists(source_path):
        shutil.move(source_path, destination_path)

# 4. Clean up the now-empty 'images' folder and annotation file
shutil.rmtree(img_dir)
os.remove(annot_file)

print("TinyImageNet validation set successfully reorganized!")

TinyImageNet validation set successfully reorganized!


In [16]:
val_dataset = ImageFolder(root="tiny-imagenet-200/val", transform=ToTensor())
print(len(val_dataset.classes))
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


200


In [20]:
print("Train classes:", dataset.class_to_idx)
print("Val classes:  ", val_dataset.class_to_idx)


Train classes: {'n01443537': 0, 'n01629819': 1, 'n01641577': 2, 'n01644900': 3, 'n01698640': 4, 'n01742172': 5, 'n01768244': 6, 'n01770393': 7, 'n01774384': 8, 'n01774750': 9, 'n01784675': 10, 'n01855672': 11, 'n01882714': 12, 'n01910747': 13, 'n01917289': 14, 'n01944390': 15, 'n01945685': 16, 'n01950731': 17, 'n01983481': 18, 'n01984695': 19, 'n02002724': 20, 'n02056570': 21, 'n02058221': 22, 'n02074367': 23, 'n02085620': 24, 'n02094433': 25, 'n02099601': 26, 'n02099712': 27, 'n02106662': 28, 'n02113799': 29, 'n02123045': 30, 'n02123394': 31, 'n02124075': 32, 'n02125311': 33, 'n02129165': 34, 'n02132136': 35, 'n02165456': 36, 'n02190166': 37, 'n02206856': 38, 'n02226429': 39, 'n02231487': 40, 'n02233338': 41, 'n02236044': 42, 'n02268443': 43, 'n02279972': 44, 'n02281406': 45, 'n02321529': 46, 'n02364673': 47, 'n02395406': 48, 'n02403003': 49, 'n02410509': 50, 'n02415577': 51, 'n02423022': 52, 'n02437312': 53, 'n02480495': 54, 'n02481823': 55, 'n02486410': 56, 'n02504458': 57, 'n025098

In [18]:
epochs = 10

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(val_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 5.241982 [    0/100000]
loss: 5.237446 [ 6400/100000]
loss: 5.232794 [12800/100000]
loss: 5.276653 [19200/100000]
loss: 5.262593 [25600/100000]
loss: 5.247660 [32000/100000]
loss: 5.272167 [38400/100000]
loss: 5.233797 [44800/100000]
loss: 5.246600 [51200/100000]
loss: 5.294264 [57600/100000]
loss: 5.267159 [64000/100000]
loss: 5.274248 [70400/100000]
loss: 5.248848 [76800/100000]
loss: 5.254525 [83200/100000]
loss: 5.259629 [89600/100000]
loss: 5.175635 [96000/100000]
Test Error: 
 Accuracy 2.1%, Avg loss: 0.081983 

Epoch 2
-------------------------------
loss: 5.224940 [    0/100000]
loss: 5.248455 [ 6400/100000]
loss: 5.248229 [12800/100000]
loss: 5.232727 [19200/100000]
loss: 5.226855 [25600/100000]
loss: 5.228253 [32000/100000]
loss: 5.203002 [38400/100000]
loss: 5.207965 [44800/100000]
loss: 5.219622 [51200/100000]
loss: 5.185251 [57600/100000]
loss: 5.219995 [64000/100000]
loss: 5.234798 [70400/100000]
loss: 5.234251 [76800/100000]


KeyboardInterrupt: 